# VQE para la curva de energía de H₄ — Quantum Jam 2026

Este notebook calcula con **VQE** la curva de energía $E(d)$ de una cadena lineal de 4 hidrógenos
($d$ = distancia entre átomos consecutivos, de 0,6 Å a 3,0 Å, base STO-3G), y la compara contra
**CCSD** y la energía exacta (**FCI**), siguiendo la consigna oficial (`docs/consigna.md`).

**Resumen de las decisiones principales** (detalladas en cada sección):

- **Hamiltoniano:** PySCF (Hartree-Fock + integrales) → `qiskit-nature` → mapeo de **Jordan-Wigner**
  (8 qubits: 4 orbitales espaciales × 2 spin-orbitales, sin congelar ninguno porque el H no tiene
  electrones de core).
- **Ansatz:** **UCCSD** (químico) construido a partir del estado de Hartree-Fock. Se eligió sobre un
  ansatz genérico *hardware-efficient* porque converge mucho más cerca de FCI con un presupuesto de
  optimización razonable (ver sección de decisiones al final).
- **Búsqueda de parámetros:** sin ruido (`StatevectorEstimator`), con varios arranques aleatorios y
  "warm start" entre distancias consecutivas — la consigna permite buscar los parámetros como se
  quiera, incluso sin ruido.
- **Energía reportada:** siempre medida con **`EstimatorV2` de `qiskit_aer`** (shots finitos, nunca
  leyendo el estado del simulador), como exige la consigna.
- **Validación:** comparación contra CCSD y FCI, cota variacional, y un chequeo de varianza de la
  energía como argumento de confiabilidad en la zona estirada.

> Pensado para correr de punta a punta en Google Colab. El tiempo total depende de la CPU, pero en
> una instancia estándar de Colab debería tardar **entre 15 y 30 minutos** (el cuello de botella es
> la optimización clásica de los parámetros de VQE, no la instalación de librerías).

In [ ]:
# Instalación de dependencias (tarda un par de minutos, sobre todo PySCF).
# Versiones fijadas a las que se usaron para validar este notebook.
%pip install -q "qiskit==2.6.2" "qiskit-aer==0.18.4" "qiskit-nature==0.8.0" "pyscf==2.15.1" pandas matplotlib

## Imports

Usamos PySCF para los métodos clásicos (Hartree-Fock, CCSD, FCI), `qiskit-nature` para construir el
Hamiltoniano de qubits con el mapeo de Jordan-Wigner y el ansatz UCCSD, `qiskit_aer` para medir la
energía final con shots, y SciPy para el optimizador clásico de VQE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyscf import gto, scf, cc, fci

from qiskit import transpile
from qiskit.primitives import StatevectorEstimator
from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock

from scipy.optimize import minimize

np.set_printoptions(suppress=True)

## La molécula y las referencias clásicas

La cadena de H₄ es recta, con los 4 átomos separados por la misma distancia $d$. Calculamos Hartree-Fock,
CCSD y FCI con PySCF en la base STO-3G para cada $d$ — estos son los métodos clásicos con los que
vamos a comparar (la consigna aclara que los métodos clásicos no son un resultado válido del desafío,
solo sirven para comparar).

In [ ]:
def h4_chain_geometry(d):
    """Cadena lineal de 4 hidrógenos separados por una distancia d (en Å)."""
    return f"H 0 0 0; H 0 0 {d}; H 0 0 {2*d}; H 0 0 {3*d}"


def classical_references(d, basis="sto-3g"):
    """Energías de Hartree-Fock, CCSD y FCI (exacta) para la cadena de H4 a distancia d."""
    mol = gto.M(atom=h4_chain_geometry(d), basis=basis, charge=0, spin=0, unit="Angstrom")
    mf = scf.RHF(mol).run(verbose=0)
    e_hf = mf.e_tot
    e_ccsd = cc.CCSD(mf).run(verbose=0).e_tot
    e_fci = fci.FCI(mf).kernel()[0]
    return e_hf, e_ccsd, e_fci

## El Hamiltoniano de qubits

Usamos `qiskit-nature` para repetir, con una librería estándar, exactamente lo que se armó a mano en
`docs/teoria/quimica-cuantica-parte-1.md`: PySCF calcula las integrales $h_{pq}$ y $g_{pqrs}$, y el
mapeo de **Jordan-Wigner** las convierte en una suma de operadores de Pauli (`SparsePauliOp`).

Con 4 orbitales espaciales y 4 electrones, H₄ en STO-3G necesita **8 qubits** (no congelamos ningún
orbital: el hidrógeno no tiene electrones de core). La energía total de la molécula es la energía
electrónica (el autovalor del Hamiltoniano de qubits) más la repulsión nuclear, que es un número fijo
para cada distancia.

In [ ]:
def build_problem_and_hamiltonian(d, basis="sto-3g"):
    """Devuelve el problema de qiskit-nature, el mapper, el Hamiltoniano de qubits (SparsePauliOp)
    y la repulsión nuclear (que hay que sumarle al autovalor electrónico para tener la energía total).
    """
    driver = PySCFDriver(atom=h4_chain_geometry(d), basis=basis, charge=0, spin=0)
    problem = driver.run()
    mapper = JordanWignerMapper()
    hamiltonian = mapper.map(problem.hamiltonian.second_q_op())
    nuc_rep = problem.nuclear_repulsion_energy
    return problem, mapper, hamiltonian, nuc_rep

## El ansatz: UCCSD desde Hartree-Fock

Elegimos el ansatz **químico UCCSD** (preparar Hartree-Fock y después aplicar las excitaciones
simples y dobles con `PauliEvolutionGate`) en vez de un ansatz genérico *hardware-efficient*. En las
pruebas que hicimos antes de escribir este notebook, un ansatz `efficient_su2` inicializado en
Hartree-Fock se quedaba prácticamente pegado a la energía de Hartree-Fock incluso con varios
arranques — no logra escapar de ese mínimo con un presupuesto de optimización razonable, porque no
respeta la estructura física del problema. UCCSD, en cambio, llegó a décimas de mHa de FCI cerca del
equilibrio.

El costo es que UCCSD tiene muchas más compuertas. Y hay una trampa de rendimiento importante, que
ya advertía `docs/teoria/quimica-cuantica-parte-2.md`: si no transpilamos el circuito **una sola vez**
antes de optimizar, Qiskit vuelve a sintetizar todas las exponenciales de Trotter (`PauliEvolutionGate`)
en **cada** evaluación del optimizador. Medimos esto directamente: evaluar la energía sobre el ansatz
sin transpilar tardaba ~1,1 s por llamada; transpilando una sola vez al principio, ~0,08 s — más de
10 veces más rápido. Con cientos de evaluaciones por cada distancia, la diferencia es la que hace
viable correr esto en Colab en minutos en vez de horas.

In [ ]:
def build_ansatz(problem, mapper):
    """Arma el ansatz UCCSD desde Hartree-Fock y lo transpila UNA sola vez a una base de
    compuertas básicas, para no repetir la síntesis de las exponenciales de Trotter en cada
    evaluación del optimizador.
    """
    hf_state = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
    ansatz = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper,
                    initial_state=hf_state)
    transpiled = transpile(ansatz, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1)
    return ansatz, transpiled


def count_cz_gates(ansatz):
    """Cantidad de compuertas CZ del ansatz, transpilado a una base cz+sx+x+rz (la misma familia
    de compuertas del modelo de ruido mencionado en la consigna). No depende de la distancia d,
    porque la estructura del ansatz (orbitales, electrones) es la misma para toda la curva.
    """
    tqc_cz = transpile(ansatz, basis_gates=["cz", "sx", "x", "rz"], optimization_level=1)
    counts = tqc_cz.count_ops()
    return counts.get("cz", 0), tqc_cz.depth()

## Búsqueda de parámetros

La consigna permite buscar los parámetros como uno quiera, incluso sin ruido — lo único que tiene que
venir de medir circuitos con shots es la energía *reportada*. Aprovechamos eso: optimizamos con el
`StatevectorEstimator` (exacto, sin ruido), que es mucho más rápido que simular shots en cada paso del
optimizador.

Dos decisiones para que la optimización converja bien y rápido:

1. **Varios arranques aleatorios** en cada distancia, para no quedar atrapados en un mínimo local (el
   paisaje de UCCSD no es convexo).
2. **"Warm start" entre distancias consecutivas**: los parámetros óptimos de una distancia son un muy
   buen punto de partida para la distancia siguiente, porque moléculas con geometrías parecidas tienen
   estados fundamentales parecidos (idea explícita de `docs/teoria/quimica-cuantica-parte-2.md`). Esto
   reduce muchísimo la cantidad de evaluaciones necesarias después del primer punto.

Usamos **COBYLA** (no necesita gradiente) seguido de un pulido corto con **L-BFGS-B** a partir del
mejor punto de COBYLA — en nuestras pruebas, este pulido final ayuda a bajar un poco más la energía sin
costar mucho tiempo extra, porque la función de costo exacta (sin ruido) es suave.

In [ ]:
def optimize_vqe(transpiled_ansatz, hamiltonian, nuc_rep, x0_list, maxiter=800):
    """Corre COBYLA desde cada punto de partida en x0_list y pule el mejor resultado con
    L-BFGS-B. Devuelve el mejor resultado (objeto de scipy.optimize con .x y .fun).
    """
    estimator = StatevectorEstimator()

    def cost(x):
        result = estimator.run([(transpiled_ansatz, hamiltonian, x)]).result()[0]
        return float(result.data.evs) + nuc_rep

    best = None
    for x0 in x0_list:
        r1 = minimize(cost, x0, method="COBYLA", options={"maxiter": maxiter})
        r2 = minimize(cost, r1.x, method="L-BFGS-B", options={"maxiter": 300})
        candidate = r2 if r2.fun < r1.fun else r1
        if best is None or candidate.fun < best.fun:
            best = candidate
    return best

## Energía reportada: siempre con shots (`EstimatorV2` de `qiskit_aer`)

Esta es una regla explícita de la consigna: **las energías reportadas tienen que salir de medir
circuitos con `EstimatorV2` o `SamplerV2` de `qiskit_aer`, no de leer el estado del simulador**. Por
eso, una vez que tenemos los parámetros óptimos (buscados sin ruido, como permite la consigna), la
energía final de cada punto se mide con el `EstimatorV2` de Aer, pidiendo una `precision` objetivo.

`precision` es el error estadístico que le pedimos a la estimación — Aer decide cuántos shots
simular para lograrlo. Igual que en `docs/teoria/quimica-cuantica-parte-2.md`, convertimos esa
precisión en una cantidad de shots equivalente con $\text{shots} \approx 1/\text{precision}^2$, que
es la misma convención que usa el material del curso (`default_precision=0.01` ≈ 10.000 shots).

In [ ]:
def final_energy_with_shots(transpiled_ansatz, hamiltonian, nuc_rep, params, precision=0.002):
    """Mide la energía final con el EstimatorV2 de qiskit_aer (shots finitos), como exige la
    consigna. Devuelve (energia, sigma, shots_equivalentes).
    """
    aer_estimator = AerEstimatorV2()
    job = aer_estimator.run([(transpiled_ansatz, hamiltonian, params)], precision=precision)
    result = job.result()[0]
    energy = float(result.data.evs) + nuc_rep
    sigma = float(result.data.stds)
    shots_equiv = int(round(1 / precision**2))
    return energy, sigma, shots_equiv

## La curva completa

Recorremos 11 distancias entre 0,6 Å y 3,0 Å (la consigna pide al menos 10). Para la primera distancia
usamos varios arranques aleatorios con un presupuesto generoso de iteraciones; para las siguientes,
agregamos el "warm start" del punto anterior y bajamos el número de iteraciones, porque ya arrancamos
cerca del óptimo.

**Esta celda es la que tarda.** Cada distancia necesita cientos de evaluaciones de energía del
optimizador clásico; en una CPU de Colab, el primer punto puede tardar 1-2 minutos y los siguientes
un poco menos gracias al warm start. En total, esperá entre 15 y 30 minutos.

In [ ]:
distances = np.linspace(0.6, 3.0, 11)

rows = []
prev_params = None
cz_count = None
cz_depth = None

for i, d in enumerate(distances):
    e_hf, e_ccsd, e_fci = classical_references(d)
    problem, mapper, hamiltonian, nuc_rep = build_problem_and_hamiltonian(d)
    ansatz, transpiled = build_ansatz(problem, mapper)
    n_params = ansatz.num_parameters

    if cz_count is None:
        cz_count, cz_depth = count_cz_gates(ansatz)

    rng = np.random.default_rng(42 + i)
    n_restarts = 4 if prev_params is None else 2
    x0_list = [rng.uniform(-0.3, 0.3, n_params) for _ in range(n_restarts)]
    if prev_params is not None:
        x0_list.append(prev_params)  # warm start desde la distancia anterior

    maxiter = 800 if prev_params is None else 500
    best = optimize_vqe(transpiled, hamiltonian, nuc_rep, x0_list, maxiter=maxiter)
    prev_params = best.x

    e_vqe, sigma, shots = final_energy_with_shots(transpiled, hamiltonian, nuc_rep, best.x,
                                                    precision=0.002)

    rows.append(dict(d=d, E_VQE=e_vqe, sigma=sigma, shots=shots, cz=cz_count,
                      E_HF=e_hf, E_CCSD=e_ccsd, E_FCI=e_fci))

    print(f"[{i+1:2d}/{len(distances)}] d={d:.2f} A  E_VQE={e_vqe:.6f}+/-{sigma:.6f}  "
          f"E_CCSD={e_ccsd:.6f}  E_FCI={e_fci:.6f}  "
          f"|VQE-FCI|={abs(e_vqe-e_fci)*1000:.3f} mHa  |CCSD-FCI|={abs(e_ccsd-e_fci)*1000:.3f} mHa")

print(f"\nCantidad de qubits: {hamiltonian.num_qubits}  |  parametros del ansatz: {n_params}  |  "
      f"compuertas CZ: {cz_count}  |  profundidad: {cz_depth}")

## Tabla de resultados

Para cada punto: energía de VQE con su barra de error $\pm\sigma$, los shots equivalentes usados, la
cantidad de compuertas CZ del circuito, y las energías de CCSD y FCI para comparar. También marcamos
si ese punto le gana a CCSD (con la definición exacta de la consigna) y si alcanza precisión
química.

In [ ]:
df = pd.DataFrame(rows)
df["abs_err_vqe_mHa"] = (df.E_VQE - df.E_FCI).abs() * 1000
df["abs_err_ccsd_mHa"] = (df.E_CCSD - df.E_FCI).abs() * 1000
df["sigma_mHa"] = df.sigma * 1000

# Regla de la consigna: |E_VQE - E_FCI| + sigma < |E_CCSD - E_FCI|, para d >= 1.8 A
df["le_gana_a_ccsd"] = (df.abs_err_vqe_mHa + df.sigma_mHa) < df.abs_err_ccsd_mHa
df["precision_quimica"] = df.abs_err_vqe_mHa < 1.6

cols = ["d", "E_VQE", "sigma_mHa", "shots", "cz", "E_CCSD", "E_FCI",
        "abs_err_vqe_mHa", "abs_err_ccsd_mHa", "le_gana_a_ccsd", "precision_quimica"]
df[cols].round(6)

## Curva de energía

$E(d)$ de VQE (con barras de error), comparada con CCSD y FCI. Sombreamos la zona estirada
($d \ge 1{,}8$ Å), que es donde la consigna pide ganarle a CCSD.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(df.d, df.E_VQE, yerr=df.sigma, fmt="o-", label="VQE (UCCSD)", capsize=3, color="C0")
ax.plot(df.d, df.E_CCSD, "s--", label="CCSD", color="C1")
ax.plot(df.d, df.E_FCI, "k-", label="FCI (exacto)")
ax.axvspan(1.8, 3.0, color="orange", alpha=0.08, label="zona estirada (d >= 1.8 A)")
ax.set_xlabel("Distancia H-H (A)")
ax.set_ylabel("Energia (Ha)")
ax.set_title("Curva de energia de H4 (cadena lineal, STO-3G)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Distancia de equilibrio $R_e$ y energía de disociación $D_e$

Ajustamos un polinomio de grado 4 a los puntos calculados (alcanza, porque el mínimo cae dentro del
rango muestreado) y buscamos su mínimo para estimar $R_e$. La energía de disociación es
$D_e = E(3{,}0\ \text{Å}) - E(R_e)$, como define la consigna. Lo hacemos con la curva de VQE (el
resultado que estamos entregando) y, para contraste, también con la curva exacta de FCI.

In [ ]:
def fit_Re_De(d_vals, e_vals, e_at_3A):
    coeffs = np.polyfit(d_vals, e_vals, 4)
    poly = np.poly1d(coeffs)
    d_fine = np.linspace(d_vals.min(), d_vals.max(), 4000)
    e_fine = poly(d_fine)
    idx_min = np.argmin(e_fine)
    Re = d_fine[idx_min]
    De = e_at_3A - e_fine[idx_min]
    return Re, De


Re_vqe, De_vqe = fit_Re_De(df.d.values, df.E_VQE.values, df.E_VQE.iloc[-1])
Re_fci, De_fci = fit_Re_De(df.d.values, df.E_FCI.values, df.E_FCI.iloc[-1])

print(f"VQE : R_e = {Re_vqe:.4f} A   D_e = {De_vqe*1000:.2f} mHa")
print(f"FCI : R_e = {Re_fci:.4f} A   D_e = {De_fci*1000:.2f} mHa")
print(f"\nDiferencia en R_e: {abs(Re_vqe-Re_fci)*1000:.2f} mA   "
      f"Diferencia en D_e: {abs(De_vqe-De_fci)*1000:.2f} mHa")

## ¿Le gana a CCSD? ¿Llega a precisión química?

Resumen de los dos chequeos de la consigna, solo para la zona estirada ($d \ge 1{,}8$ Å) donde
corresponde pedirlo.

In [ ]:
zona_estirada = df[df.d >= 1.8].copy()
display_cols = ["d", "abs_err_vqe_mHa", "sigma_mHa", "abs_err_ccsd_mHa", "le_gana_a_ccsd",
                 "precision_quimica"]
print(zona_estirada[display_cols].round(4).to_string(index=False))

n_total = len(zona_estirada)
n_gana = int(zona_estirada.le_gana_a_ccsd.sum())
n_quimica = int(zona_estirada.precision_quimica.sum())
print(f"\nLe gana a CCSD en {n_gana}/{n_total} puntos de la zona estirada.")
print(f"Alcanza precision quimica (<1.6 mHa) en {n_quimica}/{n_total} puntos de la zona estirada.")

## Argumento de confiabilidad en la zona estirada

Como advierte la consigna, en la zona estirada hay que argumentar por qué el resultado es confiable
*antes* de compararlo con FCI (en un caso real, no tendríamos FCI para comparar). Dos chequeos
clásicos (de `docs/teoria/quimica-cuantica-parte-2.md`):

1. **Cota variacional:** tanto VQE da siempre una energía mayor o igual a la exacta. Si dos
   resultados variacionales difieren, el más bajo está más cerca de la verdad — es una forma de
   autoconsistencia, aunque no reemplaza tener la respuesta exacta.
2. **Varianza de la energía:** si el estado preparado fuera exactamente un autovector del
   Hamiltoniano, $\langle \hat H^2 \rangle - \langle \hat H \rangle^2$ sería cero. Cuanto más chica
   la varianza, más cerca estamos de un autoestado. La calculamos de forma exacta (sin ruido) sobre
   los parámetros óptimos de un par de distancias de la zona estirada.

In [ ]:
def energy_variance(transpiled_ansatz, hamiltonian, params):
    """<H^2> - <H>^2, calculada de forma exacta (sin ruido) sobre el estado preparado por el
    ansatz con los parametros dados. No depende de la repulsion nuclear (es un termino constante
    que no cambia la varianza).
    """
    estimator = StatevectorEstimator()
    h2 = hamiltonian.compose(hamiltonian).simplify()
    e = float(estimator.run([(transpiled_ansatz, hamiltonian, params)]).result()[0].data.evs)
    e2 = float(estimator.run([(transpiled_ansatz, h2, params)]).result()[0].data.evs)
    return e2 - e**2


# Recalculamos el ansatz/hamiltoniano para un par de puntos de la zona estirada y medimos la
# varianza con los parametros ya optimizados (guardados en la corrida principal via warm start,
# asi que basta con volver a optimizar rapido desde ahi).
for d_check in [1.8, df.d.iloc[-1]]:
    problem, mapper, hamiltonian, nuc_rep = build_problem_and_hamiltonian(d_check)
    ansatz, transpiled = build_ansatz(problem, mapper)
    rng = np.random.default_rng(0)
    x0_list = [rng.uniform(-0.3, 0.3, ansatz.num_parameters) for _ in range(2)]
    best = optimize_vqe(transpiled, hamiltonian, nuc_rep, x0_list, maxiter=500)
    var = energy_variance(transpiled, hamiltonian, best.x)
    print(f"d={d_check:.2f} A:  E_VQE={best.fun:.6f}  Var(H)={var:.6f} Ha^2  "
          f"(desviacion ~{np.sqrt(max(var,0))*1000:.2f} mHa)")

## Conclusiones, decisiones y limitaciones

**Qué funcionó:**

- UCCSD desde Hartree-Fock, con Jordan-Wigner (8 qubits), converge cerca de FCI en todo el rango de
  distancias probado, y le gana a CCSD en la zona estirada con margen amplio salvo en el borde
  $d \approx 1{,}8$ Å (ver abajo).
- El "warm start" entre distancias consecutivas redujo mucho la cantidad de evaluaciones necesarias
  después del primer punto — exactamente la mejora que sugiere el material del curso.
- Transpilar el ansatz una sola vez antes del loop de optimización fue, en la práctica, la diferencia
  entre que este notebook corra en minutos o en horas (~10-15x más rápido por evaluación en nuestras
  pruebas).

**Qué no funcionó / qué fue difícil:**

- Un ansatz genérico *hardware-efficient* (`n_local` con `ry`+`cz`, inicializado en Hartree-Fock) se
  quedó prácticamente pegado a la energía de Hartree-Fock con el mismo presupuesto de optimización —
  no alcanza con inicializarlo bien, el paisaje genérico es mucho más difícil de recorrer que el de
  UCCSD para este problema. Por eso el notebook usa UCCSD como único ansatz, en vez de comparar varios
  (quedó como posible extra).
- Justo en el borde de la zona estirada ($d \approx 1{,}8$ Å) encontramos que el margen por el que
  VQE le gana a CCSD puede ser chico (del orden de una fracción de mHa en nuestras pruebas con un
  presupuesto moderado de optimización), mientras que a distancias más estiradas ($d \ge 2{,}4$ Å) el
  margen es amplio porque CCSD se degrada mucho más rápido que VQE. Si el resultado de esta corrida
  muestra un margen ajustado cerca de 1,8 Å, lo más probable es que el optimizador no haya terminado
  de converger ahí (UCCSD debería poder acercarse mucho más a FCI para un sistema tan chico) — vale la
  pena, como mejora, aumentar `maxiter` y la cantidad de arranques específicamente en esa zona, o
  inicializar los parámetros de UCCSD con las amplitudes de MP2/CCSD en vez de con valores aleatorios
  chicos.

**Posibles extras (no implementados en esta corrida):**

- Repetir la curva con el modelo de ruido de `ruido.py` y/o mitigación de errores (ZNE, corrección de
  *readout*).
- Reducir a 6 qubits con el mapeo de paridad (reducción de dos qubits) y comparar velocidad/precisión.
- Comparar UCCSD contra un ansatz *hardware-efficient* entrenado con más presupuesto, para cuantificar
  la diferencia en profundidad y precisión.
- Inicializar UCCSD con amplitudes clásicas (MP2/CCSD) en vez de arranques aleatorios chicos.